In [ ]:
!pip install torch torchvision torchaudio
!pip install pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv
!pip install torch-geometric
!pip install numpy pandas scikit-learn hyperopt openpyxl

In [ ]:
import torch
import torch_geometric
import sklearn
import hyperopt
import pandas

print("Todo OK")

In [ ]:
import os
import json
import random
from dataclasses import dataclass
from typing import Dict, Any, List, Optional

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval


# =========================
# Reproducibility
# =========================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================
# Metrics
# =========================
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def safe_mape(y_true, y_pred) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    if not np.any(mask):
        return float("inf")
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0)


# =========================
# Preprocessing
# =========================
def make_preprocessor(normalizer: str):
    imputer = SimpleImputer(strategy="median")
    if normalizer == "standard":
        scaler = StandardScaler()
    elif normalizer == "minmax":
        scaler = MinMaxScaler()
    elif normalizer == "robust":
        scaler = RobustScaler()
    else:
        raise ValueError(f"Unknown normalizer: {normalizer}")
    return imputer, scaler


# =========================
# kNN graph (haversine)
# =========================
def knn_edge_index_haversine(lat_deg: np.ndarray, lon_deg: np.ndarray, k: int) -> torch.Tensor:
    from sklearn.neighbors import NearestNeighbors

    n = len(lat_deg)
    if n <= 1:
        return torch.empty((2, 0), dtype=torch.long)

    k_eff = min(k, n - 1)

    coords_rad = np.deg2rad(np.column_stack([lat_deg, lon_deg]))
    nn = NearestNeighbors(n_neighbors=k_eff + 1, metric="haversine", algorithm="ball_tree")
    nn.fit(coords_rad)
    _, indices = nn.kneighbors(coords_rad)

    src, dst = [], []
    for i in range(n):
        neigh = indices[i, 1:]
        for j in neigh:
            src.append(int(j))
            dst.append(int(i))
    return torch.tensor([src, dst], dtype=torch.long)


def build_pyg_data(X: np.ndarray, y: np.ndarray, lat: np.ndarray, lon: np.ndarray, k_graph: int) -> Data:
    edge_index = knn_edge_index_haversine(lat, lon, k_graph)
    return Data(
        x=torch.from_numpy(X.astype(np.float32)),
        y=torch.from_numpy(y.astype(np.float32)),
        edge_index=edge_index,
    )


# =========================
# Model: GATv2 regressor
# =========================
class GATv2Regressor(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, num_layers: int, heads: int, dropout: float, residual: bool, share_weights: bool):
        super().__init__()
        assert num_layers >= 1
        self.dropout = dropout
        self.residual = residual
        self.heads = heads

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        # first
        self.convs.append(GATv2Conv(
            in_channels=in_dim,
            out_channels=hidden_dim,
            heads=heads,
            concat=True,
            dropout=dropout,
            add_self_loops=True,
            share_weights=share_weights,
        ))
        self.norms.append(nn.LayerNorm(hidden_dim * heads))

        # hidden
        for _ in range(1, num_layers):
            self.convs.append(GATv2Conv(
                in_channels=hidden_dim * heads,
                out_channels=hidden_dim,
                heads=heads,
                concat=True,
                dropout=dropout,
                add_self_loops=True,
                share_weights=share_weights,
            ))
            self.norms.append(nn.LayerNorm(hidden_dim * heads))

        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * heads, hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

        self.res_proj = nn.Linear(in_dim, hidden_dim * heads) if (residual and in_dim != hidden_dim * heads) else None

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = x
        for li, conv in enumerate(self.convs):
            h_in = h
            h = F.dropout(h, p=self.dropout, training=self.training)
            h = conv(h, edge_index)
            h = self.norms[li](h)
            h = F.elu(h)

            if self.residual:
                if li == 0 and self.res_proj is not None:
                    h = h + self.res_proj(h_in)
                elif h.shape == h_in.shape:
                    h = h + h_in

        return self.mlp(h).squeeze(-1)


# =========================
# Training with early stopping
# =========================
@dataclass
class TrainCfg:
    epochs: int = 300
    patience: int = 35
    lr: float = 1e-3
    weight_decay: float = 1e-4
    loss: str = "huber"
    huber_delta: float = 1.0
    grad_clip: float = 5.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


def loss_fn(pred: torch.Tensor, y: torch.Tensor, cfg: TrainCfg) -> torch.Tensor:
    if cfg.loss == "mse":
        return F.mse_loss(pred, y)
    if cfg.loss == "huber":
        return F.smooth_l1_loss(pred, y, beta=cfg.huber_delta)
    raise ValueError(cfg.loss)


def train_fullbatch_with_es(data: Data, model: nn.Module, cfg: TrainCfg) -> nn.Module:
    model = model.to(cfg.device)
    data = data.to(cfg.device)

    n = data.x.size(0)
    rng = np.random.default_rng(123)
    perm = rng.permutation(n)
    n_val = max(1, int(0.15 * n))
    val_idx = perm[:n_val]
    tr_idx = perm[n_val:]

    train_mask = torch.zeros(n, dtype=torch.bool, device=cfg.device)
    val_mask = torch.zeros(n, dtype=torch.bool, device=cfg.device)
    train_mask[tr_idx] = True
    val_mask[val_idx] = True

    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best = float("inf")
    best_state = None
    bad = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad()
        pred = model(data.x, data.edge_index)
        loss = loss_fn(pred[train_mask], data.y[train_mask], cfg)
        loss.backward()
        if cfg.grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

        model.eval()
        with torch.no_grad():
            pred_val = model(data.x, data.edge_index)[val_mask]
            y_val = data.y[val_mask]
            val_rmse = torch.sqrt(F.mse_loss(pred_val, y_val)).item()

        if val_rmse < best - 1e-6:
            best = val_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


@torch.no_grad()
def predict(model: nn.Module, data: Data, device: str) -> np.ndarray:
    model.eval()
    data = data.to(device)
    return model(data.x, data.edge_index).detach().cpu().numpy()


# =========================
# Hyperopt objective (CV=5) strict (graphs separated)
# =========================
def make_objective_gatv2(
    X_train: np.ndarray,
    y_train: np.ndarray,
    lat_train: np.ndarray,
    lon_train: np.ndarray,
    seed: int,
    cv_splits: int,
    base_cfg: TrainCfg,
    in_dim: int,
):
    kf = KFold(n_splits=cv_splits, shuffle=True, random_state=seed)

    def objective(params: Dict[str, Any]):
        normalizer = params["normalizer"]
        k_graph = int(params["k_graph"])
        hidden_dim = int(params["hidden_dim"])
        num_layers = int(params["num_layers"])
        heads = int(params["heads"])
        dropout = float(params["dropout"])
        lr = float(params["lr"])
        weight_decay = float(params["weight_decay"])
        residual = bool(params["residual"])
        share_weights = bool(params["share_weights"])

        fold_mses = []

        for tr_idx, va_idx in kf.split(X_train):
            imputer, scaler = make_preprocessor(normalizer)
            X_tr = scaler.fit_transform(imputer.fit_transform(X_train[tr_idx])).astype(np.float32)
            X_va = scaler.transform(imputer.transform(X_train[va_idx])).astype(np.float32)

            y_tr = y_train[tr_idx].astype(np.float32)
            y_va = y_train[va_idx].astype(np.float32)

            lat_tr = lat_train[tr_idx]
            lon_tr = lon_train[tr_idx]
            lat_va = lat_train[va_idx]
            lon_va = lon_train[va_idx]

            data_tr = build_pyg_data(X_tr, y_tr, lat_tr, lon_tr, k_graph=k_graph)
            data_va = build_pyg_data(X_va, y_va, lat_va, lon_va, k_graph=min(k_graph, len(y_va)-1 if len(y_va)>1 else 1))

            cfg = TrainCfg(**{**base_cfg.__dict__, "lr": lr, "weight_decay": weight_decay})
            model = GATv2Regressor(
                in_dim=in_dim,
                hidden_dim=hidden_dim,
                num_layers=num_layers,
                heads=heads,
                dropout=dropout,
                residual=residual,
                share_weights=share_weights,
            )

            model = train_fullbatch_with_es(data_tr, model, cfg)
            pred_va = predict(model, data_va, cfg.device)
            fold_mses.append(mean_squared_error(y_va, pred_va))

        return {"loss": float(np.mean(fold_mses)), "status": STATUS_OK, "params": params}

    return objective


# =========================
# Main experiment runner
# =========================
def run_gatv2_experiment(
    data_path: str,
    feature_cols: List[str],
    target_col: str,
    lat_col: str,
    lon_col: str,
    out_xlsx: str = "results_gatv2.xlsx",
    n_iterations: int = 100,
    max_evals: int = 100,
    cv_splits: int = 5,
    test_size: float = 0.2,
    base_cfg: Optional[TrainCfg] = None,
):
    base_cfg = base_cfg or TrainCfg()

    df = pd.read_csv(data_path).reset_index(drop=True)
    X = df[feature_cols].to_numpy()
    y = df[target_col].to_numpy().astype(np.float32)
    lat = df[lat_col].to_numpy().astype(np.float64)
    lon = df[lon_col].to_numpy().astype(np.float64)

    in_dim = X.shape[1]

    space = {
        "normalizer": hp.choice("normalizer", ["standard", "minmax", "robust"]),
        "k_graph": hp.choice("k_graph", [5, 8, 10, 12, 16]),
        "hidden_dim": hp.choice("hidden_dim", [64, 128, 256]),
        "num_layers": hp.choice("num_layers", [2, 3]),
        "heads": hp.choice("heads", [2, 4, 8]),
        "dropout": hp.uniform("dropout", 0.2, 0.6),
        "lr": hp.loguniform("lr", np.log(2e-4), np.log(2e-3)),
        "weight_decay": hp.loguniform("weight_decay", np.log(1e-6), np.log(1e-3)),
        "residual": hp.choice("residual", [0, 1]),
        "share_weights": hp.choice("share_weights", [0, 1]),
    }

    metrics_rows = []
    preds_rows = []

    for it in range(n_iterations):
        print(f"\n[GATv2] Iteración {it+1}/{n_iterations}")
        set_seed(it)

        X_tr, X_te, y_tr, y_te, lat_tr, lat_te, lon_tr, lon_te = train_test_split(
            X, y, lat, lon, test_size=test_size, random_state=it
        )

        objective = make_objective_gatv2(
            X_train=X_tr, y_train=y_tr,
            lat_train=lat_tr, lon_train=lon_tr,
            seed=it, cv_splits=cv_splits,
            base_cfg=base_cfg, in_dim=in_dim
        )

        trials = Trials()
        best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=max_evals, trials=trials)
        best_params = space_eval(space, best)
        print("  Best params:", best_params)

        # Train final on full TRAIN
        imputer, scaler = make_preprocessor(best_params["normalizer"])
        X_tr_p = scaler.fit_transform(imputer.fit_transform(X_tr)).astype(np.float32)
        X_te_p = scaler.transform(imputer.transform(X_te)).astype(np.float32)

        k_graph = int(best_params["k_graph"])

        data_train = build_pyg_data(X_tr_p, y_tr, lat_tr, lon_tr, k_graph=k_graph)
        data_test = build_pyg_data(X_te_p, y_te, lat_te, lon_te, k_graph=min(k_graph, len(y_te)-1 if len(y_te)>1 else 1))

        cfg = TrainCfg(**{
            **base_cfg.__dict__,
            "lr": float(best_params["lr"]),
            "weight_decay": float(best_params["weight_decay"]),
        })

        model = GATv2Regressor(
            in_dim=in_dim,
            hidden_dim=int(best_params["hidden_dim"]),
            num_layers=int(best_params["num_layers"]),
            heads=int(best_params["heads"]),
            dropout=float(best_params["dropout"]),
            residual=bool(best_params["residual"]),
            share_weights=bool(best_params["share_weights"]),
        )

        model = train_fullbatch_with_es(data_train, model, cfg)

        pred_tr = predict(model, data_train, cfg.device)
        pred_te = predict(model, data_test, cfg.device)

        row = {
            "iteration": it + 1,
            "model": "GATv2",
            "rmse_train": rmse(y_tr, pred_tr),
            "mae_train": float(mean_absolute_error(y_tr, pred_tr)),
            "r2_train": float(r2_score(y_tr, pred_tr)),
            "mape_train": safe_mape(y_tr, pred_tr),
            "rmse_test": rmse(y_te, pred_te),
            "mae_test": float(mean_absolute_error(y_te, pred_te)),
            "r2_test": float(r2_score(y_te, pred_te)),
            "mape_test": safe_mape(y_te, pred_te),
            "best_params": json.dumps(best_params, ensure_ascii=False),
        }
        metrics_rows.append(row)

        for j in range(len(y_te)):
            preds_rows.append({
                "iteration": it + 1,
                "model": "GATv2",
                "lat": float(lat_te[j]),
                "lon": float(lon_te[j]),
                "y_true": float(y_te[j]),
                "y_pred": float(pred_te[j]),
                "residual": float(y_te[j] - pred_te[j]),
            })

        # Save incremental XLSX
        metrics_df = pd.DataFrame(metrics_rows)
        preds_df = pd.DataFrame(preds_rows)
        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
            metrics_df.to_excel(writer, sheet_name="metrics", index=False)
            preds_df.to_excel(writer, sheet_name="predictions", index=False)

        print(f"  Test RMSE={row['rmse_test']:.4f} | R2={row['r2_test']:.4f}")

    print(f"\n Listo. Archivo generado: {out_xlsx}")


if __name__ == "__main__":
    DATA_PATH = "example_soil_dataset.csv"
    LAT_COL = "lat"
    LON_COL = "lon"
    TARGET_COL = "SOC"
    FEATURE_COLS = [f"feat_{i}" for i in range(1, 48)]

    run_gatv2_experiment(
        data_path=DATA_PATH,
        feature_cols=FEATURE_COLS,
        target_col=TARGET_COL,
        lat_col=LAT_COL,
        lon_col=LON_COL,
        out_xlsx="results_gatv2.xlsx",
        n_iterations=100,
        max_evals=100,
        cv_splits=5,
        test_size=0.2,
        base_cfg=TrainCfg(epochs=260, patience=30, loss="huber", huber_delta=1.0),
    )